# CIFAR-100 ResNet-18 Table 2/3 Reproduction Collector, fixed aggregation

This notebook collects the outputs from the CIFAR-100 R18 reproduction SLURM runs and analyzes them using the same validation-selection protocol as `analyze_v4.ipynb`.

The fixes in this version are important:

1. `eval_seeds` from `config.yaml` is **not** used as a grouping key. The actual linear-probe seed is read from `linear_eval_results_v2.csv`.
2. Linear probing is aggregated hierarchically: average probe seeds within each pretrained encoder seed, then compute the mean and t-interval across encoder seeds.
3. kNN is aggregated hierarchically: average split seeds within each pretrained encoder seed, then compute the mean and t-interval across encoder seeds.
4. Model selection is restricted to complete configurations by default, requiring four pretrained encoder seeds. This avoids selecting one-seed WEINCE rows with zero confidence intervals.
5. Result CSVs are read robustly without mutating the original files.


In [ ]:
from __future__ import annotations

import glob
import os
import re
from math import sqrt
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import yaml
from scipy.stats import t
from IPython.display import display

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


## 1. Configure paths

Edit these paths if your SLURM output directories differ. The notebook assumes it is run from the repo root, but absolute paths work too.


In [ ]:
BASELINE_DIR = Path("outputs_cifar100_r18_baseline_repro")
WEINCE_DIR = Path("outputs_policy_tlambda_select_cifar100_r18_repro")
OUT_DIR = Path("table23_r18_repro_notebook_summary")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET = "CIFAR100"
RESNET = "resnet18"

# Paper protocol for the R18 reproduction.
EXPECTED_ENCODER_SEEDS = 4
EXPECTED_LINEAR_PROBE_SEEDS = 5
EXPECTED_KNN_SPLIT_SEEDS = 5
REQUIRE_COMPLETE_CONFIGS = True

print("baseline dir:", BASELINE_DIR.resolve())
print("weince dir:  ", WEINCE_DIR.resolve())
print("out dir:     ", OUT_DIR.resolve())
print("require complete configs:", REQUIRE_COMPLETE_CONFIGS)


## 2. Helper functions

These helpers are adapted from `analyze_v4.ipynb`, but the grouping logic is stricter:

- `pretrain_seed` is read from `config.yaml` key `seed`.
- `eval_seed` is read from the linear evaluation CSV.
- `split_seed` is read from the kNN evaluation CSV.
- Config key `eval_seeds` is ignored for aggregation.
- Hyperparameter selection only considers complete four-encoder configurations unless `REQUIRE_COMPLETE_CONFIGS=False`.


In [ ]:
LINEAR_HEADER = [
    "seed",
    "epoch_num",
    "eval_epochs",
    "val loss",
    "val accuracy",
    "test loss",
    "test accuracy",
]

# Columns that define a training configuration.  The pretraining seed is
# intentionally excluded and handled separately as `pretrain_seed`.
HYPERPARAM_COLS = [
    "dataset", "resnet", "policy", "use_weib_topm",
    "weib_lambda", "weib_top_m", "auto_weib_beta", "weib_beta",
    "use_pl_topm", "pl_lambda", "pl_top_m",
    "select_aic_margin", "select_kappa_rho", "select_kappa_aic",
    "select_q", "select_tail_frac", "select_min_tail_points", "select_weib_top_m",
    "anchor_tail_frac", "anchor_min_tail_points", "anchor_r2_min", "shrink_c",
]

# The notebook chooses the best hyperparameter configuration within these method groups.
METHOD_GROUP_COLS = ["dataset", "resnet", "policy", "use_weib_topm"]


def normalize_config_value(value):
    """Make YAML values groupby-safe and stable."""
    if isinstance(value, (list, tuple)):
        return tuple(value)
    if isinstance(value, np.generic):
        return value.item()
    return value


def normalize_dataset_name(value):
    if pd.isna(value):
        return value
    return str(value).upper()


def normalize_resnet_name(value):
    if pd.isna(value):
        return value
    return str(value).lower()


def read_linear_csv(filename: str | os.PathLike[str]) -> pd.DataFrame:
    """Read a linear_eval_results_v2.csv without modifying it on disk."""
    filename = str(filename)
    try:
        df = pd.read_csv(filename, index_col=None, header=0)
    except Exception:
        df = pd.read_csv(filename, index_col=None, header=None)

    # If the CSV had no usable header, reinterpret it with the known schema.
    has_expected = set(["seed", "val accuracy", "test accuracy"]).issubset(set(df.columns))
    if not has_expected:
        raw = pd.read_csv(filename, index_col=None, header=None)
        if raw.shape[1] == len(LINEAR_HEADER):
            raw.columns = LINEAR_HEADER
            # Drop an accidentally read header row if present.
            if str(raw.iloc[0]["seed"]).strip().lower() == "seed":
                raw = raw.iloc[1:].reset_index(drop=True)
            df = raw
        else:
            raise ValueError(f"Unexpected linear CSV shape {raw.shape} for {filename}")

    return df


def read_result_csv(filename: str | os.PathLike[str], result_type: str) -> pd.DataFrame:
    if result_type == "linear":
        df = read_linear_csv(filename)
        df = df.rename(columns={"seed": "eval_seed", "eval seed": "eval_seed"})
    else:
        df = pd.read_csv(filename, index_col=None, header=0)
    return df


def get_dataframe(
    basedir: str | os.PathLike[str],
    result_type: str = "linear",
) -> pd.DataFrame:
    """Read all result CSVs under basedir and attach selected config.yaml fields."""
    basedir = Path(basedir)
    pattern = basedir / f"**/{result_type}_eval_results_v2.csv"
    all_files = sorted(glob.glob(str(pattern), recursive=True))
    frames: list[pd.DataFrame] = []

    for filename in all_files:
        try:
            df = read_result_csv(filename, result_type=result_type)
        except Exception as exc:
            print(f"[warn] failed to read {filename}: {exc}")
            continue
        if df.empty:
            continue

        run_dir = Path(filename).parent
        config_path = run_dir / "config.yaml"
        if not config_path.exists():
            print(f"[warn] missing config.yaml next to {filename}")
            continue
        with open(config_path, "r", encoding="utf-8") as f:
            config = yaml.safe_load(f) or {}

        # Critical fix: do not add config['eval_seeds'] as a grouping column.
        # The actual probe/split seed is already recorded in the result CSV.
        df["pretrain_seed"] = config.get("seed", np.nan)
        for key in HYPERPARAM_COLS:
            if key == "seed":
                continue
            if key in config:
                df[key] = normalize_config_value(config.get(key))
            elif key not in df.columns:
                df[key] = np.nan

        if "dataset" in df.columns:
            df["dataset"] = df["dataset"].map(normalize_dataset_name)
        if "resnet" in df.columns:
            df["resnet"] = df["resnet"].map(normalize_resnet_name)
        if "use_weib_topm" in df.columns:
            # Preserve NaN, but normalize actual booleans/strings.
            df["use_weib_topm"] = df["use_weib_topm"].map(
                lambda x: x if pd.isna(x) else (str(x).lower() == "true" if isinstance(x, str) else bool(x))
            )

        df["run_dir"] = str(run_dir)
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    frame = pd.concat(frames, axis=0, ignore_index=True)
    for col in frame.columns:
        if any(token in col for token in ["val", "test", "loss", "accuracy", "recall"]):
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    for col in ["pretrain_seed", "eval_seed", "split_seed"]:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    return frame


def t_interval_halfwidth(values: np.ndarray, alpha: float = 0.05) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n <= 1:
        return 0.0
    se = values.std(ddof=1) / sqrt(n)
    return float(t.ppf(1 - alpha / 2, n - 1) * se)


def aggregate_hierarchical(
    df: pd.DataFrame,
    metrics: list[str],
    eval_seed_col: str,
    config_cols: list[str] | None = None,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """Average eval/split seeds within encoder, then summarize across encoder seeds."""
    if df.empty:
        return pd.DataFrame()
    config_cols = list(config_cols or HYPERPARAM_COLS)
    for col in config_cols + ["pretrain_seed"]:
        if col not in df.columns:
            df[col] = np.nan
    if eval_seed_col not in df.columns:
        df[eval_seed_col] = 0

    metrics = [m for m in metrics if m in df.columns]
    if not metrics:
        return pd.DataFrame()

    # First level: probe/split-seed mean inside each pretrained encoder seed.
    per_encoder_cols = config_cols + ["pretrain_seed"]
    per_encoder = (
        df.groupby(per_encoder_cols, dropna=False)[metrics]
        .mean()
        .reset_index()
    )

    # Second level: t interval across pretrained encoder seeds.
    rows = []
    for key, g in per_encoder.groupby(config_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        row = {col: key[i] for i, col in enumerate(config_cols)}
        seeds = sorted([int(s) for s in g["pretrain_seed"].dropna().unique()])
        row["pretrain_seed_values"] = ",".join(map(str, seeds))
        row["n_unique_pretrain_seeds"] = len(seeds)
        for metric in metrics:
            vals = g[metric].astype(float).dropna().values
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_std_enc"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
            row[f"{metric}_n_enc"] = int(len(vals))
            row[f"{metric}_ci_halfwidth"] = t_interval_halfwidth(vals, alpha=alpha)
        rows.append(row)
    return pd.DataFrame(rows)


def require_complete(summary_df: pd.DataFrame, metric_mean_col: str, expected_n: int = EXPECTED_ENCODER_SEEDS) -> pd.DataFrame:
    """Filter to complete four-encoder configurations for the selection metric."""
    if summary_df.empty or not REQUIRE_COMPLETE_CONFIGS:
        return summary_df
    if not metric_mean_col.endswith("_mean"):
        return summary_df
    metric = metric_mean_col[: -len("_mean")]
    n_col = f"{metric}_n_enc"
    if n_col not in summary_df.columns:
        return summary_df
    complete = summary_df[summary_df[n_col] >= expected_n].copy()
    if complete.empty:
        print(f"[warn] no complete configs for {metric}; falling back to all rows")
        return summary_df
    dropped = len(summary_df) - len(complete)
    if dropped > 0:
        print(f"[info] dropped {dropped} incomplete configs for selection on {metric}")
    return complete


def choose_best_val_model(summary_df: pd.DataFrame, val_metric_mean_col: str, group_cols: list[str]) -> pd.DataFrame:
    """Within each method group, select the row maximizing a validation metric."""
    if summary_df.empty:
        return pd.DataFrame()
    df = summary_df.copy()
    for col in group_cols:
        if col not in df.columns:
            df[col] = np.nan
    df = require_complete(df, val_metric_mean_col)
    best_rows = []
    for _, group in df.groupby(group_cols, dropna=False):
        valid = group.dropna(subset=[val_metric_mean_col])
        if valid.empty:
            continue
        best_rows.append(valid.loc[valid[val_metric_mean_col].idxmax()])
    return pd.DataFrame(best_rows).reset_index(drop=True)


def summarize_linear_all(base_dirs: list[str | os.PathLike[str]]) -> pd.DataFrame:
    dfs = [get_dataframe(base_dir, result_type="linear") for base_dir in base_dirs]
    dfs = [df for df in dfs if not df.empty]
    if not dfs:
        return pd.DataFrame()
    full_df = pd.concat(dfs, axis=0, ignore_index=True)
    metrics = ["test accuracy", "val accuracy", "test loss", "val loss"]
    return aggregate_hierarchical(full_df, metrics=metrics, eval_seed_col="eval_seed")


def summarize_linear_selected(base_dirs: list[str | os.PathLike[str]]) -> pd.DataFrame:
    summary = summarize_linear_all(base_dirs)
    return choose_best_val_model(summary, "val accuracy_mean", group_cols=METHOD_GROUP_COLS)


## 3. Check run completeness

This section gives a quick sanity check before aggregating.


In [ ]:
def summarize_run_status(base_dir: Path) -> pd.DataFrame:
    rows = []
    for run_dir in sorted([p for p in base_dir.glob("**/") if p.is_dir()]):
        cfg = run_dir / "config.yaml"
        if not cfg.exists():
            continue
        try:
            config = yaml.safe_load(cfg.read_text()) or {}
        except Exception:
            config = {}
        rows.append({
            "run_dir": str(run_dir),
            "dataset": config.get("dataset"),
            "resnet": config.get("resnet"),
            "policy": config.get("policy"),
            "seed": config.get("seed"),
            "select_aic_margin": config.get("select_aic_margin"),
            "select_kappa_rho": config.get("select_kappa_rho"),
            "select_kappa_aic": config.get("select_kappa_aic"),
            "has_checkpoint": (run_dir / "checkpoint_100.tar").exists(),
            "has_linear": (run_dir / "linear_eval_results_v2.csv").exists(),
            "has_knn": (run_dir / "knn_eval_results_v2.csv").exists(),
        })
    return pd.DataFrame(rows)

baseline_status = summarize_run_status(BASELINE_DIR)
weince_status = summarize_run_status(WEINCE_DIR)

print("baseline run count:", len(baseline_status))
print("weince run count:  ", len(weince_status))

display(baseline_status.head())
display(weince_status.head())

if not baseline_status.empty:
    display(baseline_status[["has_checkpoint", "has_linear", "has_knn"]].mean().rename("baseline_fraction_complete"))
if not weince_status.empty:
    display(weince_status[["has_checkpoint", "has_linear", "has_knn"]].mean().rename("weince_fraction_complete"))

baseline_status.to_csv(OUT_DIR / "baseline_run_status.csv", index=False)
weince_status.to_csv(OUT_DIR / "weince_run_status.csv", index=False)


## 4. Linear-probe aggregation, Table 2 style

This reproduces the linear side of `analyze_v4.ipynb`: choose the best hyperparameter configuration by validation accuracy, then report the corresponding test accuracy.


In [ ]:
columns_to_show = [
    "dataset", "resnet", "policy", "use_weib_topm",
    "select_aic_margin", "select_kappa_rho", "select_kappa_aic",
    "test accuracy_mean", "test accuracy_ci_halfwidth",
    "val accuracy_mean", "val accuracy_ci_halfwidth", "test accuracy_n_enc",
    "pretrain_seed_values",
]

baseline_linear_all = summarize_linear_all([BASELINE_DIR])
weince_linear_all = summarize_linear_all([WEINCE_DIR])

baseline_linear = choose_best_val_model(baseline_linear_all, "val accuracy_mean", METHOD_GROUP_COLS)
if not baseline_linear.empty and "use_weib_topm" in baseline_linear:
    baseline_linear = baseline_linear[baseline_linear["use_weib_topm"] == False]

weince_linear = choose_best_val_model(weince_linear_all, "val accuracy_mean", METHOD_GROUP_COLS)

combined_linear = pd.concat([baseline_linear, weince_linear], axis=0, ignore_index=True)
if not combined_linear.empty:
    combined_linear = combined_linear[
        (combined_linear["dataset"] == DATASET) &
        (combined_linear["resnet"] == RESNET)
    ]

linear_cols = [c for c in columns_to_show if c in combined_linear.columns]
display(combined_linear[linear_cols])
combined_linear.to_csv(OUT_DIR / "r18_linear_selected_full.csv", index=False)
combined_linear[linear_cols].to_csv(OUT_DIR / "r18_linear_selected_table.csv", index=False)


In [ ]:
def table2_percent(linear_df: pd.DataFrame) -> pd.DataFrame:
    if linear_df.empty:
        return pd.DataFrame()
    rows = []
    for _, row in linear_df.iterrows():
        method = "WEINCE" if row.get("policy") == "tlambda_select" else "InfoNCE"
        rows.append({
            "Dataset": row.get("dataset"),
            "Encoder": row.get("resnet"),
            "Method": method,
            "Linear Acc (%)": 100 * row.get("test accuracy_mean", np.nan),
            "Linear CI95 halfwidth (%)": 100 * row.get("test accuracy_ci_halfwidth", np.nan),
            "Val Acc (%)": 100 * row.get("val accuracy_mean", np.nan),
            "Val CI95 halfwidth (%)": 100 * row.get("val accuracy_ci_halfwidth", np.nan),
            "n_encoder_seeds": row.get("test accuracy_n_enc", np.nan),
            "pretrain_seed_values": row.get("pretrain_seed_values", ""),
            "select_aic_margin": row.get("select_aic_margin", np.nan),
            "select_kappa_rho": row.get("select_kappa_rho", np.nan),
            "select_kappa_aic": row.get("select_kappa_aic", np.nan),
        })
    out = pd.DataFrame(rows)
    out.to_csv(OUT_DIR / "table2_r18_percent.csv", index=False)
    return out

table2_r18 = table2_percent(combined_linear)
display(table2_r18)


## 5. kNN aggregation, Table 3 style

This follows `analyze_v4.ipynb`: for each validation metric, choose the best configuration and report the matched test metric. This means R@1, R@2, etc. may select different WEINCE hyperparameters.


In [ ]:
def col_order(col: str) -> str:
    name_parts = col.split("_")
    if len(name_parts) >= 3 and name_parts[-2] == "ci":
        return f"{name_parts[2]}_{int(name_parts[-3]):3}z"
    if len(name_parts) >= 2 and name_parts[-2].isdigit():
        return f"{name_parts[2]}_{int(name_parts[-2]):3}"
    return col


def summarize_knn(base_dir: str | os.PathLike[str]) -> pd.DataFrame:
    df = get_dataframe(base_dir, result_type="knn")
    if df.empty:
        return pd.DataFrame()

    val_columns = [col for col in df.columns if col.startswith("val_")]
    test_cols = {col[5:]: col for col in df.columns if col.startswith("test_")}
    best_summaries = []

    for val_col in val_columns:
        suffix = val_col[4:]  # remove 'val_'
        if suffix not in test_cols:
            continue
        test_col = test_cols[suffix]
        summary = aggregate_hierarchical(
            df,
            metrics=[val_col, test_col],
            eval_seed_col="split_seed",
        )
        best_sum = choose_best_val_model(
            summary,
            val_metric_mean_col=f"{val_col}_mean",
            group_cols=METHOD_GROUP_COLS,
        )
        if best_sum.empty:
            continue
        best_sum["selected_by"] = val_col
        best_sum["selected_for"] = test_col
        best_summaries.append(best_sum)

    if not best_summaries:
        return pd.DataFrame()
    return pd.concat(best_summaries, axis=0, ignore_index=True)


def compact_knn(best: pd.DataFrame) -> pd.DataFrame:
    if best.empty:
        return pd.DataFrame()
    rows = []
    for keys, group in best.groupby(METHOD_GROUP_COLS, dropna=False):
        row = dict(zip(METHOD_GROUP_COLS, keys))
        for _, r in group.iterrows():
            metric = r["selected_for"]
            row[f"{metric}_mean"] = r.get(f"{metric}_mean", np.nan)
            row[f"{metric}_ci_halfwidth"] = r.get(f"{metric}_ci_halfwidth", np.nan)
            row[f"{metric}_n_enc"] = r.get(f"{metric}_n_enc", np.nan)
            row[f"{metric}_selected_by"] = r.get("selected_by")
            row[f"{metric}_pretrain_seed_values"] = r.get("pretrain_seed_values", "")
            for hp in ["select_aic_margin", "select_kappa_rho", "select_kappa_aic"]:
                if hp in r.index:
                    row[f"{metric}_{hp}"] = r.get(hp, np.nan)
        rows.append(row)
    return pd.DataFrame(rows)

baseline_knn_long = summarize_knn(BASELINE_DIR)
if not baseline_knn_long.empty and "use_weib_topm" in baseline_knn_long:
    baseline_knn_long = baseline_knn_long[baseline_knn_long["use_weib_topm"] == False]

weince_knn_long = summarize_knn(WEINCE_DIR)
combined_knn_long = pd.concat([baseline_knn_long, weince_knn_long], axis=0, ignore_index=True)
if not combined_knn_long.empty:
    combined_knn_long = combined_knn_long[
        (combined_knn_long["dataset"] == DATASET) &
        (combined_knn_long["resnet"] == RESNET)
    ]

combined_knn_compact = compact_knn(combined_knn_long)

display(combined_knn_long)
display(combined_knn_compact)

combined_knn_long.to_csv(OUT_DIR / "r18_knn_selected_long.csv", index=False)
combined_knn_compact.to_csv(OUT_DIR / "r18_knn_selected_compact.csv", index=False)


In [ ]:
KNN_METRICS = [
    "test_knn_accuracy_50",
    "test_knn_recall_at_1",
    "test_knn_recall_at_2",
    "test_knn_recall_at_5",
    "test_knn_recall_at_10",
    "test_knn_recall_at_20",
    "test_knn_recall_at_50",
]


def table3_percent(compact_df: pd.DataFrame) -> pd.DataFrame:
    if compact_df.empty:
        return pd.DataFrame()
    rows = []
    for _, row in compact_df.iterrows():
        method = "WEINCE" if row.get("policy") == "tlambda_select" else "InfoNCE"
        out = {
            "Dataset": row.get("dataset"),
            "Encoder": row.get("resnet"),
            "Method": method,
        }
        for metric in KNN_METRICS:
            mean_col = f"{metric}_mean"
            ci_col = f"{metric}_ci_halfwidth"
            n_col = f"{metric}_n_enc"
            if mean_col in row:
                out[f"{metric} (%)"] = 100 * row.get(mean_col, np.nan)
                out[f"{metric} CI95 halfwidth (%)"] = 100 * row.get(ci_col, np.nan)
                out[f"{metric} n_encoder_seeds"] = row.get(n_col, np.nan)
        rows.append(out)
    out = pd.DataFrame(rows)
    out.to_csv(OUT_DIR / "table3_r18_percent.csv", index=False)
    return out

table3_r18 = table3_percent(combined_knn_compact)
display(table3_r18)


## 6. Compare against the reported CIFAR-100 R18 rows

The targets below are the CIFAR-100 R18 entries in the paper: Table 2 for linear accuracy, Table 3 for R@1 through R@20, and Appendix Tables 6–7 for kNN accuracy and R@50. The paper protocol uses validation-only model selection, four pretrained encoder seeds, five linear-probe seeds per encoder, and five kNN split seeds.


In [ ]:
TARGET_LINEAR = {
    ("InfoNCE", "resnet18"): {"linear": 50.01, "ci": 0.19},
    ("WEINCE", "resnet18"): {"linear": 53.28, "ci": 0.28},
}

TARGET_KNN = {
    ("InfoNCE", "resnet18"): {
        "test_knn_accuracy_50": 40.73,
        "test_knn_recall_at_1": 36.98,
        "test_knn_recall_at_2": 47.42,
        "test_knn_recall_at_5": 61.72,
        "test_knn_recall_at_10": 72.18,
        "test_knn_recall_at_20": 81.45,
        "test_knn_recall_at_50": 90.75,
    },
    ("WEINCE", "resnet18"): {
        "test_knn_accuracy_50": 47.38,
        "test_knn_recall_at_1": 42.94,
        "test_knn_recall_at_2": 53.29,
        "test_knn_recall_at_5": 66.19,
        "test_knn_recall_at_10": 76.11,
        "test_knn_recall_at_20": 84.41,
        "test_knn_recall_at_50": 92.81,
    },
}

# Linear comparison.
lin_cmp = []
for _, row in table2_r18.iterrows():
    method = row["Method"]
    key = (method, row["Encoder"])
    target = TARGET_LINEAR.get(key, {})
    lin_cmp.append({
        "Method": method,
        "Encoder": row["Encoder"],
        "reproduced_linear": row["Linear Acc (%)"],
        "reproduced_ci": row.get("Linear CI95 halfwidth (%)", np.nan),
        "target_linear": target.get("linear", np.nan),
        "target_ci": target.get("ci", np.nan),
        "delta_linear": row["Linear Acc (%)"] - target.get("linear", np.nan),
        "n_encoder_seeds": row.get("n_encoder_seeds", np.nan),
    })
lin_cmp = pd.DataFrame(lin_cmp)
display(lin_cmp)
lin_cmp.to_csv(OUT_DIR / "linear_vs_reported_table2.csv", index=False)

# kNN comparison.
knn_cmp = []
for _, row in table3_r18.iterrows():
    method = row["Method"]
    key = (method, row["Encoder"])
    target = TARGET_KNN.get(key, {})
    out = {"Method": method, "Encoder": row["Encoder"]}
    for metric in KNN_METRICS:
        col = f"{metric} (%)"
        ci_col = f"{metric} CI95 halfwidth (%)"
        n_col = f"{metric} n_encoder_seeds"
        if col in row:
            out[f"{metric}_reproduced"] = row[col]
            out[f"{metric}_ci"] = row.get(ci_col, np.nan)
            out[f"{metric}_target"] = target.get(metric, np.nan)
            out[f"{metric}_delta"] = row[col] - target.get(metric, np.nan)
            out[f"{metric}_n_encoder_seeds"] = row.get(n_col, np.nan)
    knn_cmp.append(out)
knn_cmp = pd.DataFrame(knn_cmp)
display(knn_cmp)
knn_cmp.to_csv(OUT_DIR / "knn_vs_reported_table3.csv", index=False)


## 7. WEINCE hyperparameter diagnostics

These diagnostics are not needed for Tables 2–3, but they are useful for seeing which `tlambda_select` settings were selected by validation.


In [ ]:
all_weince_linear = weince_linear_all.copy()
if not all_weince_linear.empty:
    all_weince_linear = all_weince_linear[
        (all_weince_linear["dataset"] == DATASET) &
        (all_weince_linear["resnet"] == RESNET)
    ].copy()
    all_weince_linear["complete_for_linear_selection"] = all_weince_linear["val accuracy_n_enc"] >= EXPECTED_ENCODER_SEEDS
    diag_cols = [
        "policy", "select_aic_margin", "select_kappa_rho", "select_kappa_aic",
        "test accuracy_mean", "test accuracy_ci_halfwidth",
        "val accuracy_mean", "val accuracy_ci_halfwidth",
        "test accuracy_n_enc", "val accuracy_n_enc", "complete_for_linear_selection",
        "pretrain_seed_values",
    ]
    diag_cols = [c for c in diag_cols if c in all_weince_linear.columns]
    display(all_weince_linear[diag_cols].sort_values("val accuracy_mean", ascending=False).head(20))
    all_weince_linear[diag_cols].sort_values("val accuracy_mean", ascending=False).to_csv(
        OUT_DIR / "weince_linear_grid_ranked.csv", index=False
    )
else:
    print("No WEINCE linear results found yet.")


In [ ]:
if plt is not None and not all_weince_linear.empty:
    # Simple visualization: best validation accuracy for each (margin, rho), taking max over kappa_aic.
    pivot = all_weince_linear.pivot_table(
        index="select_aic_margin",
        columns="select_kappa_rho",
        values="val accuracy_mean",
        aggfunc="max",
    )
    display(pivot)
    ax = pivot.plot(marker="o", figsize=(7, 4))
    ax.set_xlabel("select_aic_margin")
    ax.set_ylabel("best validation linear accuracy")
    ax.set_title("WEINCE R18 CIFAR-100 validation accuracy")
    ax.grid(True)
    plt.tight_layout()
    fig_path = OUT_DIR / "weince_linear_val_by_margin_rho.pdf"
    plt.savefig(fig_path)
    print("saved", fig_path)
    plt.show()


## 8. Final CSV outputs

The notebook writes all key outputs to `OUT_DIR`:

```text
baseline_run_status.csv
weince_run_status.csv
r18_linear_selected_table.csv
table2_r18_percent.csv
r18_knn_selected_long.csv
r18_knn_selected_compact.csv
table3_r18_percent.csv
linear_vs_reported_table2.csv
knn_vs_reported_table3.csv
weince_linear_grid_ranked.csv
```


In [ ]:
print("Wrote outputs to:", OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)
